In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "stock_data.db"
TRADING_DAYS = 252

conn = sqlite3.connect(DB_PATH)

# Equal-weighted sector portfolios. vw_portfolio_overview averages each sector's
# constituent daily returns, which implicitly rebalances back to equal weight
# every single day — a deliberate portfolio choice, not a neutral one, and not
# what a buy-and-hold investor actually experiences.
sectors = pd.read_sql("""
    SELECT Date, Sector, sector_avg_daily_return AS ret, tickers_counted
    FROM vw_portfolio_overview
    ORDER BY Date, Sector
""", conn, parse_dates=['Date'])

spy = pd.read_sql("""
    SELECT Date, Benchmark_return AS ret
    FROM benchmark_data
    WHERE Benchmark_return IS NOT NULL
    ORDER BY Date
""", conn, parse_dates=['Date'])

conn.close()

# One column per sector plus SPY, one row per trading day.
panel = sectors.pivot(index='Date', columns='Sector', values='ret')
panel['SPY'] = spy.set_index('Date')['ret']

print("panel shape:", panel.shape)
print("date range:", panel.index.min().date(), "→", panel.index.max().date())
print("\nnulls per column:")
print(panel.isna().sum())
print("\nconstituents per sector per day:")
print(sectors['tickers_counted'].value_counts())

panel shape: (1667, 12)
date range: 2020-01-03 → 2026-08-21

nulls per column:
Sector
Communication Services    0
Consumer Discretionary    0
Consumer Staples          0
Energy                    0
Financials                0
Health Care               0
Industrials               0
Information Technology    0
Materials                 0
Real Estate               0
Utilities                 0
SPY                       0
dtype: int64

constituents per sector per day:
tickers_counted
10    18337
Name: count, dtype: int64


In [2]:
def summarize(returns: pd.Series) -> dict:
    """Performance stats for one daily-return series.

    Sharpe here uses a ZERO risk-free rate, matching the existing Power BI /
    Tableau measure. That is a real assumption: with cash yielding several
    percent over much of this window, a zero-rf Sharpe flatters everything
    equally. Fine for ranking, wrong as an absolute number. Say so out loud.
    """
    r = returns.dropna()
    total = (1 + r).prod() - 1
    years = (r.index[-1] - r.index[0]).days / 365.25
    return {
        'total_return': total,
        'cagr': (1 + total) ** (1 / years) - 1,
        'ann_vol': r.std(ddof=0) * np.sqrt(TRADING_DAYS),
        'sharpe': r.mean() / r.std(ddof=0) * np.sqrt(TRADING_DAYS),
    }

summary = pd.DataFrame({c: summarize(panel[c]) for c in panel.columns}).T
summary = summary.sort_values('total_return', ascending=False)

spy = summary.loc['SPY']
summary['beat_spy_return'] = summary['total_return'] > spy['total_return']
summary['beat_spy_sharpe'] = summary['sharpe'] > spy['sharpe']

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print(summary)

                        total_return   cagr  ann_vol  sharpe  beat_spy_return  \
Information Technology        4.3790 0.2888   0.2816  1.0445             True   
Energy                        2.1034 0.1862   0.3805  0.6424             True   
Financials                    1.9059 0.1745   0.3000  0.6875             True   
Industrials                   1.8567 0.1715   0.2414  0.7784             True   
Materials                     1.8125 0.1688   0.2345  0.7843             True   
Consumer Discretionary        1.6976 0.1614   0.2442  0.7373             True   
Health Care                   1.5879 0.1542   0.1883  0.8576             True   
SPY                           1.5858 0.1540   0.2017  0.8135            False   
Consumer Staples              0.9988 0.1101   0.1644  0.7190            False   
Communication Services        0.8558 0.0977   0.2061  0.5570            False   
Real Estate                   0.7730 0.0902   0.2406  0.4815            False   
Utilities                   

In [3]:
sector_cols = [c for c in panel.columns if c != 'SPY']

# Equal-weighted portfolio of all 110 names. Because every sector holds exactly
# 10 tickers, weighting all 110 equally is identical to averaging the 11 sector
# portfolios — so this is just the row-wise mean.
panel['EW110'] = panel[sector_cols].mean(axis=1)

summary = pd.DataFrame({c: summarize(panel[c]) for c in panel.columns}).T
summary = summary.sort_values('total_return', ascending=False)

for bench in ['SPY', 'EW110']:
    b = summary.loc[bench]
    summary[f'beats_{bench}_ret'] = summary['total_return'] > b['total_return']
    summary[f'beats_{bench}_sharpe'] = summary['sharpe'] > b['sharpe']

print(summary)

                        total_return   cagr  ann_vol  sharpe  beats_SPY_ret  \
Information Technology        4.3790 0.2888   0.2816  1.0445           True   
Energy                        2.1034 0.1862   0.3805  0.6424           True   
Financials                    1.9059 0.1745   0.3000  0.6875           True   
Industrials                   1.8567 0.1715   0.2414  0.7784           True   
Materials                     1.8125 0.1688   0.2345  0.7843           True   
EW110                         1.7793 0.1667   0.1925  0.8999           True   
Consumer Discretionary        1.6976 0.1614   0.2442  0.7373           True   
Health Care                   1.5879 0.1542   0.1883  0.8576           True   
SPY                           1.5858 0.1540   0.2017  0.8135          False   
Consumer Staples              0.9988 0.1101   0.1644  0.7190          False   
Communication Services        0.8558 0.0977   0.2061  0.5570          False   
Real Estate                   0.7730 0.0902   0.2406

In [4]:
from itertools import combinations

SPLIT = '2024-01-01'
is_p  = panel[panel.index <  SPLIT]
oos_p = panel[panel.index >= SPLIT]

print("in-sample:    ", is_p.index.min().date(), "→", is_p.index.max().date(), f"({len(is_p)} days)")
print("out-of-sample:", oos_p.index.min().date(), "→", oos_p.index.max().date(), f"({len(oos_p)} days)")

def stats(r):
    return (1 + r).prod() - 1, r.mean() / r.std(ddof=0) * np.sqrt(TRADING_DAYS)

rows = []
for k in range(1, len(sector_cols) + 1):
    for combo in combinations(sector_cols, k):
        c = list(combo)
        is_t,  is_s  = stats(is_p[c].mean(axis=1))
        oos_t, oos_s = stats(oos_p[c].mean(axis=1))
        rows.append({'n': k, 'sectors': ' + '.join(c),
                     'is_total': is_t, 'is_sharpe': is_s,
                     'oos_total': oos_t, 'oos_sharpe': oos_s})

combos = pd.DataFrame(rows)
print(f"\n{len(combos)} combinations evaluated\n")

bench = pd.DataFrame({
    b: {'is_total': stats(is_p[b])[0],  'is_sharpe': stats(is_p[b])[1],
        'oos_total': stats(oos_p[b])[0], 'oos_sharpe': stats(oos_p[b])[1]}
    for b in ['SPY', 'EW110']
}).T
print(bench)

in-sample:     2020-01-03 → 2023-12-29 (1005 days)
out-of-sample: 2024-01-02 → 2026-08-21 (662 days)

2047 combinations evaluated

       is_total  is_sharpe  oos_total  oos_sharpe
SPY      0.5581     0.6055     0.6596      1.3042
EW110    0.7387     0.7257     0.5985      1.5131


In [6]:
combos['is_rank']  = combos['is_sharpe'].rank(ascending=False)
combos['oos_rank'] = combos['oos_sharpe'].rank(ascending=False)

rho = combos['is_sharpe'].rank().corr(combos['oos_sharpe'].rank())
print(f"Spearman rank correlation, in-sample vs out-of-sample Sharpe: {rho:.3f}")
print(f"(1.0 = in-sample ranking perfectly predicts out-of-sample; 0.0 = no information)\n")

top10 = combos.nsmallest(10, 'is_rank')[
    ['n', 'sectors', 'is_sharpe', 'is_rank', 'oos_sharpe', 'oos_rank']
]
print("Top 10 combinations chosen on IN-SAMPLE Sharpe, and where they landed out-of-sample:")
print(top10.to_string(index=False))

ew_oos = bench.loc['EW110', 'oos_sharpe']
n_beat = (top10['oos_sharpe'] > ew_oos).sum()
print(f"\nOf the 10 in-sample winners, {n_beat}/10 beat EW110 out-of-sample "
      f"(EW110 OOS Sharpe = {ew_oos:.4f})")
print(f"Median out-of-sample rank of the in-sample top 10: {top10['oos_rank'].median():.0f} of {len(combos)}")

Spearman rank correlation, in-sample vs out-of-sample Sharpe: 0.117
(1.0 = in-sample ranking perfectly predicts out-of-sample; 0.0 = no information)

Top 10 combinations chosen on IN-SAMPLE Sharpe, and where they landed out-of-sample:
 n                                                                          sectors  is_sharpe  is_rank  oos_sharpe   oos_rank
 2                                             Health Care + Information Technology     1.0061   1.0000      1.3953 1,190.0000
 1                                                           Information Technology     0.9891   2.0000      1.1574 1,924.0000
 3                    Consumer Discretionary + Health Care + Information Technology     0.9799   3.0000      1.2257 1,828.0000
 3                          Consumer Staples + Health Care + Information Technology     0.9569   4.0000      1.5617   239.0000
 4 Consumer Discretionary + Consumer Staples + Health Care + Information Technology     0.9567   5.0000      1.3543 1,401.0000
 4 

In [7]:
winners = [s for s in sector_cols if summary.loc[s, 'beats_EW110_ret']]
losers  = [s for s in sector_cols if not summary.loc[s, 'beats_EW110_ret']]

corr = panel[sector_cols].corr()

def avg_pairwise(cols):
    sub = corr.loc[cols, cols].values
    return sub[np.triu_indices(len(cols), k=1)].mean()

print("Sectors that beat EW110 on total return:", winners, "\n")
print(f"avg pairwise correlation — winners:     {avg_pairwise(winners):.3f}")
print(f"avg pairwise correlation — non-winners: {avg_pairwise(losers):.3f}")
print(f"avg pairwise correlation — all 11:      {avg_pairwise(sector_cols):.3f}\n")
print(corr.loc[winners, winners].round(3))

Sectors that beat EW110 on total return: ['Energy', 'Financials', 'Industrials', 'Information Technology', 'Materials'] 

avg pairwise correlation — winners:     0.646
avg pairwise correlation — non-winners: 0.601
avg pairwise correlation — all 11:      0.578

Sector                  Energy  Financials  Industrials  \
Sector                                                    
Energy                  1.0000      0.6490       0.6630   
Financials              0.6490      1.0000       0.8190   
Industrials             0.6630      0.8190       1.0000   
Information Technology  0.3840      0.5940       0.6010   
Materials               0.5630      0.7530       0.8160   

Sector                  Information Technology  Materials  
Sector                                                     
Energy                                  0.3840     0.5630  
Financials                              0.5940     0.7530  
Industrials                             0.6010     0.8160  
Information Technology   